# 🚀 MotionAI Coach – Google Colab Pose Server (GVHMR)

Notebook dựng server trích xuất **pose 3D thật** cho backend (FAQ Q5: được host pipeline AI lên Colab):

```
backend (máy bạn) --AI_REMOTE_URL--> Cloudflare tunnel --> pose_server (cổng 8100) --> GVHMR-Demo API (cổng 8000, GPU)
```

- `pose_server` (của dự án) nhận video, gọi GVHMR-Demo, đọc `hmr4d_results.pt` rồi trả về 24 khớp SMPL.
- Chạy theo kiểu **job bất đồng bộ** (`POST /jobs` rồi hỏi `GET /jobs/{id}`) vì GVHMR mất vài phút / video,
  vượt giới hạn ~100 giây mỗi request của Cloudflare tunnel.
- Đặt `MODEL = "yolo"` ở Bước 2 nếu chỉ muốn chạy nhanh bằng YOLO pose (không cần checkpoint GVHMR).

### ⚠️ Trước tiên: đổi sang GPU
**Runtime → Change runtime type → T4 GPU → Save.**

### Bước 1: Kiểm tra GPU

In [ ]:
!nvidia-smi || echo '⚠️ Đang chạy CPU. Vào Runtime -> Change runtime type -> chọn T4 GPU!'

### Bước 2: Cấu hình & tải mã nguồn dự án
`CKPT_DRIVE_DIR` là thư mục trên Google Drive giữ checkpoint GVHMR để lần sau không phải tải lại.

In [ ]:
import os, secrets

MODEL = "gvhmr"                     # "gvhmr" (pose 3D thật) hoặc "yolo" (nhanh, pose 2D nâng lên 3D)
PROJECT_REPO = "https://github.com/trunghieu714/projectCNWed.git"
GVHMR_REPO = "https://github.com/doantrunghieu08/GVHMR-Demo.git"
GVHMR_DIR = "/content/GVHMR"
CKPT_DRIVE_DIR = "/content/drive/MyDrive/GVHMR_checkpoints"
STATIC_CAM = True                   # điện thoại đặt cố định trên chân máy khi quay
GVHMR_API_KEY = secrets.token_hex(16)   # khóa nội bộ giữa pose_server và GVHMR-Demo
POSE_SERVER_KEY = secrets.token_hex(16) # khóa backend phải gửi kèm (AI_REMOTE_API_KEY)

!pip install -q fastapi uvicorn python-multipart httpx opencv-python-headless scipy pyngrok gdown
if os.path.exists('/content/projectCNWed'):
    !cd /content/projectCNWed && git pull -q
else:
    !git clone -q {PROJECT_REPO} /content/projectCNWed
print('✅ Đã tải mã nguồn dự án')

### Bước 3 (GVHMR): Cài GVHMR-Demo (~2–3 phút)
`requirements.txt` của GVHMR-Demo ghim torch 2.3 / numpy 1.23 / ultralytics 8.2 cho Python 3.10. Trên Python 3.13 của Colab,
numpy 1.x không có bản dựng sẵn nên phải biên dịch rất lâu. Ô này giữ **torch và numpy 2 có sẵn của Colab** và dùng
ultralytics mới (cùng API tracking), nên **không gói nào phải biên dịch**; cài bằng `uv` (nhanh hơn pip nhiều lần).
Chạy lại ô trong cùng runtime sẽ bỏ qua bước cài.

**pytorch3d**: GVHMR chỉ thật sự dùng `pytorch3d.transforms` (PyTorch thuần); phần C++/CUDA (`knn`, `renderer`) chỉ
được import để render video – bước GVHMR-Demo đã bỏ. Ô này chép mã Python của pytorch3d **không biên dịch**, kèm
module `_C` giả (gọi tới sẽ báo lỗi rõ ràng).

Dòng đỏ `dependency conflicts` về các gói khác của Colab (jax, geopandas...) thì bỏ qua được.
GVHMR-Demo chạy trong tiến trình riêng (Bước 5) nên **không cần khởi động lại runtime** sau khi cài.

In [ ]:
import glob, re, subprocess
DEPS_MARKER = '/content/.gvhmr_deps_ok'
if MODEL == "gvhmr":
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(CKPT_DRIVE_DIR, exist_ok=True)

    if not os.path.exists(GVHMR_DIR):
        !git clone -q --depth 1 {GVHMR_REPO} {GVHMR_DIR}

    if os.path.exists(DEPS_MARKER):
        print('⏭️ Thư viện GVHMR đã cài trong runtime này – bỏ qua')
    else:
        # Bỏ các dòng ghim chỉ chạy trên Python 3.10; chumpy chỉ dùng khi đánh giá với SMPL .pkl, không cần cho suy luận
        skip = re.compile(r'^(--extra-index-url|torch==|torchvision==|numpy==|ultralytics==|imageio==|pytorch3d|cython_bbox|chumpy|ipdb|black)')
        with open(f'{GVHMR_DIR}/requirements.txt', encoding='utf-8') as f:
            reqs = [l.strip() for l in f if l.strip() and not l.startswith('#') and not skip.match(l.strip())]
        with open('/content/gvhmr_requirements.txt', 'w') as f:
            f.write('\n'.join(reqs + ['ultralytics>=8.3', 'imageio>=2.34', 'pytorch-lightning==2.3.0', 'iopath']))
        !pip install -q uv
        !uv pip install --system -q -r /content/gvhmr_requirements.txt || pip install -q -r /content/gvhmr_requirements.txt
        !cd {GVHMR_DIR} && uv pip install --system -q --no-deps -e .

        # pytorch3d: chép mã Python (không biên dịch) + module _C giả
        site_dir = subprocess.check_output(['python', '-c', "import sysconfig; print(sysconfig.get_paths()['purelib'])"], text=True).strip()
        !rm -rf /content/pytorch3d_src {site_dir}/pytorch3d
        !git clone -q --depth 1 --branch stable https://github.com/facebookresearch/pytorch3d.git /content/pytorch3d_src
        !cp -r /content/pytorch3d_src/pytorch3d {site_dir}/pytorch3d
        with open(f'{site_dir}/pytorch3d/_C.py', 'w', encoding='utf-8') as f:
            f.write('# Thay cho phần mở rộng C++/CUDA của pytorch3d (không biên dịch được với torch của Colab).\n'
                    '# GVHMR chỉ dùng pytorch3d.transforms; knn/renderer chỉ được import, không gọi khi bỏ render.\n'
                    '# Hằng số lấy từ pytorch3d/csrc/ext.cpp (pulsar dùng làm giá trị mặc định lúc import).\n'
                    'EPS = 1e-6\nMAX_FLOAT = 3.4e38\nMAX_INT = 2147483647\nMAX_UINT = 4294967295\n'
                    'MAX_USHORT = 65535\nPULSAR_MAX_GRAD_SPHERES = 128\n\n'
                    'def __getattr__(name):\n'
                    '    raise AttributeError(f"pytorch3d._C.{name} không có: pytorch3d được cài không kèm phần mở rộng C++/CUDA")\n')

    # Kiểm tra: import đúng chuỗi module mà GVHMR-Demo dùng khi chạy job
    ok = subprocess.run(['python', '-c', "import numpy, cv2, torch, app.services.video_service; print('✅ GVHMR import OK | numpy', numpy.__version__, '| opencv', cv2.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())"],
                        cwd=GVHMR_DIR, env={**os.environ, 'PYTHONPATH': GVHMR_DIR}, capture_output=True, text=True)
    print(ok.stdout or ok.stderr[-3000:])
    if ok.returncode == 0:
        open(DEPS_MARKER, 'w').close()
else:
    print('Bỏ qua (MODEL = yolo)')

### Bước 4 (GVHMR): Checkpoint
Cấu trúc cần có trong `MyDrive/GVHMR_checkpoints/` (ô dưới sẽ liên kết thư mục này vào `GVHMR/inputs/checkpoints`):

```
body_models/smplx/SMPLX_NEUTRAL.npz   <- BẮT BUỘC tự tải: đăng ký https://smpl-x.is.tue.mpg.de/
body_models/smpl/SMPL_NEUTRAL.pkl     <- tự tải: đăng ký https://smpl.is.tue.mpg.de/
gvhmr/gvhmr_siga24_release.ckpt       156 MB  ┐
hmr2/epoch=10-step=25000.ckpt         2,5 GB  │ ô dưới tự lấy (tải về nghĩa là bạn đồng ý license của từng mô hình)
vitpose/vitpose-h-multi-coco.pth      2,4 GB  │
yolo/yolov8x.pt                       131 MB  ┘ <- lấy từ GitHub chính thức của Ultralytics
```
Thứ tự ô dưới thử lấy từng file (chỉ một lần – sau đó file nằm trên Drive của bạn):
1. **Lối tắt Drive** tới thư mục chính thức – cách nên dùng khi Drive báo *"Too many users have viewed or downloaded"*:
   mở [thư mục checkpoint chính thức của GVHMR](https://drive.google.com/drive/folders/1eebJ13FUEXrKBawHpJroW0sNSxLjh9xD),
   bấm **⋮ → Sắp xếp → Thêm lối tắt (Add shortcut)** → chọn **Drive của tôi**, rồi chạy lại ô này.
2. `gdown` từng file từ Drive chính thức (hay bị giới hạn lượt tải theo ngày).
3. Bản sao trên HuggingFace (`camenduru/GVHMR`) – **chỉ khi bạn đặt `ALLOW_HF_MIRROR = True`**. Đây là bản do bên thứ ba đăng lại,
   cùng dung lượng với bản chính thức; file `.ckpt/.pth` là pickle (nạp vào là chạy code), nên ô ghim đúng một commit và
   kiểm tra SHA-256 – nhưng mã băm chỉ chứng minh file không bị đổi so với commit đó, không chứng minh đó là bản gốc của tác giả.

SMPL/SMPL-X có license riêng nên không được phân phối lại – mỗi người phải tự đăng ký và tải, rồi upload lên Drive.

In [ ]:
import hashlib, shutil
ALLOW_HF_MIRROR = False   # True = được tải từ bản sao HuggingFace nếu Drive chính thức hết lượt (đọc chú thích ở trên)

if MODEL == "gvhmr":
    OFFICIAL_IDS = {   # id từng file trong thư mục Drive chính thức của GVHMR
        "gvhmr/gvhmr_siga24_release.ckpt": "1c9iCeKFN4Kr6cMPJ9Ss6Jdc3SZFnO5NP",
        "hmr2/epoch=10-step=25000.ckpt": "1X5hvVqvqI9tvjUCb2oAlZxtgIKD9kvsc",
        "vitpose/vitpose-h-multi-coco.pth": "1sR8xZD9wrZczdDVo6zKscNLwvarIRhP5",
        "yolo/yolov8x.pt": "1_HGm-lqIH83-M1ML4bAXaqhm_eT2FKo5",
    }
    YOLO_URL = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8x.pt"
    HF_MIRROR = "https://huggingface.co/camenduru/GVHMR/resolve/21b32d5389e2e59c0737d4c4095bbc0b8c23f66b"
    HF_SHA256 = {
        "gvhmr/gvhmr_siga24_release.ckpt": "4fae7da2de388d5da3514cb27a2d003f364dacb280e9cf88972b710e589c6b91",
        "hmr2/epoch=10-step=25000.ckpt": "2dcf79638109781d1ae5f5c44fee5f55bc83291c210653feead9b7f04fa6f20e",
        "vitpose/vitpose-h-multi-coco.pth": "50e33f4077ef2a6bcfd7110c58742b24c5859b7798fb0eedd6d2215e0a8980bc",
        "yolo/yolov8x.pt": "c4d5a3f000d771762f03fc8b57ebd0aae324aeaefdd6e68492a9c4470f2d1e8b",
    }
    BODY_MODELS = ["body_models/smplx/SMPLX_NEUTRAL.npz", "body_models/smpl/SMPL_NEUTRAL.pkl"]
    target = lambda p: f'{CKPT_DRIVE_DIR}/{p}'
    have = lambda p: os.path.exists(target(p)) and os.path.getsize(target(p)) > 1_000_000

    def sha256(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(8 << 20), b''):
                h.update(chunk)
        return h.hexdigest()

    def put(tmp, p):
        os.makedirs(os.path.dirname(target(p)), exist_ok=True)
        shutil.move(tmp, target(p))

    def find_in_drive(name):
        # Lối tắt tới thư mục chính thức (Colab hiện nội dung lối tắt ở .shortcut-targets-by-id) hoặc file "Bản sao của ..."
        roots = ['/content/drive/.shortcut-targets-by-id', '/content/drive/MyDrive']
        for root in roots:
            for dirpath, dirnames, filenames in os.walk(root):
                if dirpath.startswith(CKPT_DRIVE_DIR) or dirpath.count('/') - root.count('/') > 3:
                    dirnames[:] = []
                    continue
                for fn in filenames:
                    if fn == name or fn.endswith(' ' + name):
                        return os.path.join(dirpath, fn)
        return None

    for p in OFFICIAL_IDS:
        if have(p):
            continue
        name = os.path.basename(p)
        tmp = f'/content/ckpt_dl_{name}'
        if p.startswith('yolo/'):
            print('⏳', p, '<- GitHub Ultralytics')
            !wget -q -O "{tmp}" {YOLO_URL}
            if os.path.exists(tmp) and os.path.getsize(tmp) > 1_000_000:
                put(tmp, p); continue
        src = find_in_drive(name)
        if src:
            print('⏳', p, '<- lối tắt / bản sao trên Drive:', src)
            !cp "{src}" "{tmp}"
            if os.path.exists(tmp) and os.path.getsize(tmp) > 1_000_000:
                put(tmp, p); continue
        print('⏳', p, '<- gdown Drive chính thức')
        !gdown -q {OFFICIAL_IDS[p]} -O "{tmp}"
        if os.path.exists(tmp) and os.path.getsize(tmp) > 1_000_000:
            put(tmp, p); continue
        if ALLOW_HF_MIRROR:
            print('⏳', p, '<- bản sao HuggingFace (camenduru/GVHMR @ 21b32d5)')
            !wget -q -O "{tmp}" "{HF_MIRROR}/{p}"
            if os.path.exists(tmp) and sha256(tmp) == HF_SHA256[p]:
                put(tmp, p); continue
            print('   ❌ Sai SHA-256 hoặc tải lỗi – đã bỏ file')
        if os.path.exists(tmp):
            os.remove(tmp)

    !mkdir -p {GVHMR_DIR}/inputs && rm -rf {GVHMR_DIR}/inputs/checkpoints && ln -s {CKPT_DRIVE_DIR} {GVHMR_DIR}/inputs/checkpoints
    print()
    for p in BODY_MODELS + list(OFFICIAL_IDS):
        print(('✅ ' if have(p) else '❌ THIẾU ') + p)
    if not all(have(p) for p in BODY_MODELS + list(OFFICIAL_IDS)):
        print()
        print('⚠️ Còn thiếu file. Cách xử lý (chọn 1):')
        print('   a) Thêm lối tắt thư mục chính thức vào Drive của tôi (xem chú thích ở trên) rồi chạy lại ô này;')
        print('   b) đợi vài giờ cho Drive hết giới hạn lượt tải rồi chạy lại;')
        print('   c) đặt ALLOW_HF_MIRROR = True (bản sao bên thứ ba, có kiểm tra SHA-256) rồi chạy lại.')
else:
    print('Bỏ qua (MODEL = yolo)')

### Bước 5 (GVHMR): Khởi động GVHMR-Demo API (cổng 8000, chỉ trong Colab)
Mô hình nạp vào GPU mất khoảng 1–3 phút; ô này đợi tới khi log báo `Model loaded successfully`.

In [ ]:
import subprocess, time
if MODEL == "gvhmr":
    !fuser -k 8000/tcp 2>/dev/null || true
    gvhmr_log = open('/content/gvhmr.log', 'w')
    gvhmr_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8000'],
        cwd=GVHMR_DIR, env={**os.environ, 'API_KEY': GVHMR_API_KEY, 'PYTHONPATH': GVHMR_DIR, 'PYTHONUNBUFFERED': '1'},
        stdout=gvhmr_log, stderr=subprocess.STDOUT)
    for _ in range(300):
        time.sleep(2)
        log = open('/content/gvhmr.log', errors='ignore').read()
        if 'Model loaded successfully' in log or gvhmr_proc.poll() is not None or 'Không tìm thấy checkpoint' in log:
            break
    print(log[-3000:])
    print('✅ GVHMR-Demo đã sẵn sàng' if 'Model loaded successfully' in log else '❌ GVHMR chưa nạp được mô hình – xem log ở trên')
else:
    print('Bỏ qua (MODEL = yolo)')

### Bước 6: Khởi động pose_server của dự án (cổng 8100)

In [ ]:
import httpx
!fuser -k 8100/tcp 2>/dev/null || true
env = {**os.environ, 'PYTHONPATH': '/content/projectCNWed/backend', 'POSE_SERVER_KEY': POSE_SERVER_KEY, 'PYTHONUNBUFFERED': '1'}
if MODEL == "gvhmr":
    env.update(POSE_BACKEND='gvhmr_api', GVHMR_URL='http://127.0.0.1:8000', GVHMR_API_KEY=GVHMR_API_KEY,
               GVHMR_DIR=GVHMR_DIR, GVHMR_STATIC_CAM='1' if STATIC_CAM else '0', POSE_TIMEOUT='1800')
else:
    !pip install -q ultralytics
    env.update(POSE_BACKEND='yolo', YOLO_POSE_WEIGHTS='yolo11x-pose.pt')
pose_log = open('/content/pose_server.log', 'w')
pose_proc = subprocess.Popen(['python', '-m', 'ai_engine.remote_server.pose_server'], cwd='/content/projectCNWed/backend',
                             env=env, stdout=pose_log, stderr=subprocess.STDOUT)
time.sleep(4)
try:
    print('✅ pose_server online:', httpx.get('http://127.0.0.1:8100/', timeout=5).json())
except Exception as e:
    print('❌ Lỗi:', e); print(open('/content/pose_server.log').read()[-2000:])

### Bước 7: Chạy thử với video mẫu (khuyên làm trước khi demo)
Gửi `docs/example_video/tennis.mp4` của GVHMR-Demo qua đúng giao thức job mà backend sẽ dùng.

In [ ]:
sample = f'{GVHMR_DIR}/docs/example_video/tennis.mp4'
if not os.path.exists(sample):
    !wget -q -O /content/tennis.mp4 https://raw.githubusercontent.com/doantrunghieu08/GVHMR-Demo/main/docs/example_video/tennis.mp4
    sample = '/content/tennis.mp4'
hdr = {'X-API-Key': POSE_SERVER_KEY}
with open(sample, 'rb') as f:
    job_id = httpx.post('http://127.0.0.1:8100/jobs', headers=hdr, files={'file': ('tennis.mp4', f, 'video/mp4')}, timeout=120).json()['job_id']
t0 = time.time()
while True:
    job = httpx.get(f'http://127.0.0.1:8100/jobs/{job_id}', headers=hdr, timeout=30).json()
    print(f"\r[{time.time()-t0:5.0f}s] {job['status']}: {job.get('progress')}", end='')
    if job['status'] in ('COMPLETED', 'FAILED'):
        break
    time.sleep(5)
print()
if job['status'] == 'COMPLETED':
    r = job['result']
    print(f"✅ {len(r['joints_3d'])} frame x {len(r['joints_3d'][0])} khớp, nguồn {r['source']}, {r['seconds']}s")
else:
    print('❌', job.get('error'))
    for log in ('/content/pose_server.log', '/content/gvhmr.log'):
        if os.path.exists(log):
            print(f'--- {log}'); print(open(log, errors='ignore').read()[-2000:])

### Bước 8: Mở tunnel ra Internet (Cloudflare) và lấy cấu hình cho backend

In [ ]:
!which cloudflared > /dev/null || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null)
!killall cloudflared 2>/dev/null || true
logfile = '/content/cloudflared.log'
if os.path.exists(logfile):
    os.remove(logfile)
!nohup cloudflared tunnel --url http://127.0.0.1:8100 --logfile /content/cloudflared.log > /dev/null 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(logfile):
        m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', open(logfile, errors='ignore').read())
        if m:
            tunnel_url = m[-1]
            break

def print_env(url):
    print('=' * 75)
    print('📋 DÁN VÀO backend/.env TRÊN MÁY BẠN rồi khởi động lại backend (và Celery worker nếu có):')
    print(f'AI_REMOTE_URL={url}')
    print(f'AI_REMOTE_API_KEY={POSE_SERVER_KEY}')
    print('AI_ESTIMATOR_TYPE=remote,precomputed,mock')
    print('=' * 75)

if tunnel_url:
    print_env(tunnel_url)
else:
    print('⚠️ Chưa lấy được URL Cloudflare – dùng Ngrok ở ô dưới. Log:')
    print(open(logfile, errors='ignore').read() if os.path.exists(logfile) else '')

### (Tùy chọn) Bước 9: Ngrok nếu mạng chặn Cloudflare

In [ ]:
from pyngrok import ngrok
NGROK_TOKEN = ''  # token miễn phí: https://dashboard.ngrok.com/get-started/your-authtoken
if NGROK_TOKEN.strip():
    ngrok.kill(); ngrok.set_auth_token(NGROK_TOKEN.strip())
    print_env(ngrok.connect(8100).public_url)
else:
    print('Bỏ qua nếu Bước 8 đã in ra link Cloudflare.')

### Khi gặp lỗi
- Log GVHMR: `!tail -50 /content/gvhmr.log` · Log pose_server: `!tail -50 /content/pose_server.log`
- Colab ngắt kết nối sau ~90 phút không tương tác / 12 giờ: chạy lại từ Bước 2 (checkpoint & wheel pytorch3d đã nằm trên Drive). URL tunnel đổi mỗi lần → cập nhật lại `.env`.
- Backend vẫn chấm được nếu Colab tắt: `AI_ESTIMATOR_TYPE` là chuỗi dự phòng, nguồn pose thực tế được ghi trong kết quả chấm.